# Lesson 5, Part 4

In [ ]:
# initializations
import numpy as np
from numpy.polynomial import Polynomial as P
import matplotlib.pyplot as plt

def newton_raphson(x0, fun, deriv, tol=1e-14):
    x = x0
    if deriv == None:
        deriv = lambda z: 1000 * (fun(z + 0.001) - fun(z))
    for n in range(100):
        y = fun(x)
        if abs(y) < tol:
            break
        x -= y / deriv(x)
    return x

Let's code a Taylor series for exp(*x*) using the Polynomial class from numpy.

In [ ]:
def taylor_exp(order):
    inv_factorial = 1
    coef = [inv_factorial]
    for n in range(1,order+1):
        inv_factorial /=  n
        coef.append(inv_factorial)
    return P(coef)

print(taylor_exp(16))

Here are plots of various Taylor series approximations to exp(*x*).

In [ ]:
x = np.linspace(-3,3,100)
plt.title('Taylor Approximations to Exponential')
for n in range(1,10):
    p = taylor_exp(n)
    plt.plot(x,p(x),label='order '+str(n))
plt.legend()
plt.show()

Let's examine the convergence to exp(1).

In [ ]:
print('Taylor approximations for exp(1)')
for n in range(1,20):
    p = taylor_exp(n)
    print('order',n,':',p(1))

How far do the Taylor series approximations diverge from the identity exp(*x*)&sdot;exp(-*x*) = 1? 
This code plots the relative divergence and prints its maximum values.

In [ ]:
x = np.linspace(0,4,100)
print('Maximum relative divergence')
for n in range(5,20):
    p = taylor_exp(n) 
    divergence = np.abs(p(x) * p(-x) - 1)
    rel_diverge = divergence / p(x)
    plt.plot(x,rel_diverge,label='order '+str(n))
    print('order',n,':',round(np.max(rel_diverge),8))
plt.title('Relative divergences from exp(x)*exp(-x)=1')
plt.legend()
plt.show()

Let's turn to the natural logarithm ln *x*, defined as the integral of 1/*z* from 1 to *x*.
We can estimate values using Simpson's rule.

In [ ]:
def simpson_ln(x,num=100):
    num = 2 * (num//2) + 1  # need odd number of points
    x_pts = np.linspace(1,x,num)
    dx = (x - 1) / (num - 1)
    y_pts = 1 / x_pts
    weight = np.zeros(num)
    weight[0] = weight[-1] = 1/3
    weight[1:-1:2] = 4/3
    weight[2:-2:2] = 2/3 
    wtd_sum_y = np.sum(y_pts * weight)
    return wtd_sum_y * dx

for num in [6,12,25,50,100,200,400,800]:
    print('num =',num,': ln(0.2) ~',simpson_ln(0.2,num),', ln(2) ~',simpson_ln(2,num),', ln(20) ~',simpson_ln(20,num))
x = np.linspace(0.2,20,200)
y = [simpson_ln(val) for val in x]
plt.title('Simpson approximation to ln(x)')
plt.plot(x,y)
plt.show()

If we transpose the previous chart, it closely resembles exp(*x*).
This is not an accident, as we will see in Level 6.

In [ ]:
x = np.linspace(0.2,20,200)
y = [simpson_ln(val) for val in x]
plt.title('x as a function of Simpson approximation to ln(x)')
plt.plot(y,x)
plt.show()

Let's now consider Taylor approximations for the natural logarithm.
Here is the basic Taylor series for ln(*x*) = ln(1 + *z*) around *z* = *x* - 1 = 0.

In [ ]:
def taylor_ln(order,x=None):
    coef = np.zeros(order+1)
    for n in range(1, order+1):
        coef[n] = 1/n if n % 2 else -1/n
    if x is None:
        return P(coef)
    if isinstance(x, list):
        x = np.array(x)
    return P(coef)(x-1) 
    
print('ln(1+x) =',taylor_ln(12))

Unfortunately, convergence can be slow, and fails completely for *x*>2.

In [ ]:
x = [0.2,1.8,2.6]
m = 3
print('Algorithm   ln',x[0],'   ln',x[1],'  ln',x[2])
for order in range(3,13):
    p = taylor_ln(order,x)
    print('Taylor',order,': ',round(p[0],m),'  ',round(p[1],m),'  ',round(p[2],m))
print('Simpson   : ',round(simpson_ln(x[0]),m),'  ',round(simpson_ln(x[1]),m),'  ',round(simpson_ln(x[2]),m))

Since ln *x* = -ln(1/*x*), applying taylor_ln to the right-hand side gives a convergent series in *w* = 1 - 1/*x* for *x* > 1/2.

In [ ]:
def taylor_ln_alt(order,x=None):
    Q = -taylor_ln(order)
    if x is None:
         return Q.coef
    if isinstance(x,list):
        x = np.array(x)
    return Q(1/x-1) 

x2=2
print('approx ln(2): basic     alt Taylor')
for order in range(3,20):
    basic = taylor_ln(order,x2)
    alt = taylor_ln_alt(order,x2)
    print('Taylor',order,':   ',round(basic,4),'  ',round(alt,7))

If we combine taylor_ln for *x* <= 1  with taylor_ln_alt for *x*>1, the fit approaches Simpson's but convergence remains slow for large *x*.

In [ ]:
def taylor_ln_comb(order,x=None):
    coef = np.zeros(order+1)
    for n in range(1,order+1):
        coef[n] = 1/n if n % 2 else -1/n
    Pbasic = P(coef)
    if x is None:
         return Pbasic
    results = [ ]
    for val in x:
        if  val <= 1:
            results.append(Pbasic(val-1))
        else:
            results.append(-Pbasic(1/val-1))                 
    return np.array(results)

x = np.linspace(0.2,16,200)
y = [simpson_ln(val) for val in x]
z10 = taylor_ln_comb(10,x)
z20 = taylor_ln_comb(20,x)
plt.title('Simpson and combined Taylor approximations for ln(x)')
plt.plot(x,y,label='Simpson')
plt.plot(x,z10,label='Comb Taylor 10')
plt.plot(x,z20,label='Comb Taylor 20')
plt.legend()
plt.show()

A faster-converging series for ln *x*, which works for all *x*, 
relies on odd powers of *v* = (*x* - 1) / (*x* + 1), since ln *x* = ln(1 + *v*) - ln(1 - *v*).

In [ ]:
def taylor_ln_better(order,x=None):
    coef = np.zeros(order+1)
    for k in range(1,order+1,2):
        coef[k] = 2 / k 
    if x is None:
        return P(coef)
    if isinstance(x,list):
        x = np.array(x)
    return P(coef)((x-1)/(x+1)) 

x = np.linspace(0.2,16,200)
y = [simpson_ln(val) for val in x]
zbetter10 = taylor_ln_better(10,x)
zbetter20 = taylor_ln_better(20,x)
plt.title('Simpson and better Taylor approximations for ln(x)')
plt.plot(x,y,x,zbetter10,x,zbetter20)
plt.legend(['Simpson','Better Taylor 10','Better Taylor 20'])
plt.show()
print('v =',taylor_ln_better(20))

Here are Taylor approximations for cosine. 
The higher the order, the more oscillations between -1 and +1 they generate before veering off.

In [ ]:
def taylor_cos(order,x=None):
    coef = np.zeros(order+1)
    coef[0] = 1
    for k in range(2,order+1,2):
        coef[k] = -coef[k-2] / (k * (k-1))  
    if x is None:
        return P(coef)
    if isinstance(x,list):
        x = np.array(x)
    return P(coef)(x)

print(taylor_cos(24))
plt.title("Taylor approximations for cosine")
x = np.linspace(-9,9,200)
plt.ylim(-1.5,1.5)     # this limits the y axis to the interval -1.5 to +1.5 
for order in range(2,20,2):
    plt.plot(x,taylor_cos(order,x),label='order '+str(order))
plt.legend()

plt.figure()
plt.title("Taylor approximation of order 50 for cosine")
x = np.linspace(-21,21,200)
plt.ylim(-1.2,1.2)  
plt.plot(x,taylor_cos(50,x))
plt.plot(x,np.zeros(len(x)),':')   # ':' plots a dotted line
plt.show()

Here are Taylor approximations for sine. 
In the central region they resemble shifted versions of the cosine approximations.

In [ ]:
def taylor_sin(order,x=None):
    coef = np.zeros(order+1)
    coef[1] = 1
    for k in range(3,order+1,2):
        coef[k] = -coef[k-2] / (k * (k-1))  
    if x is None:
        return P(coef)
    if isinstance(x,list):
        x = np.array(x)
    return P(coef)(x) 

print(taylor_sin(24))
plt.title("Taylor approximations for sine")
x = np.linspace(-9,9,200)
plt.ylim(-1.5,1.5)  
for order in range(1,20,2):
    plt.plot(x,taylor_sin(order,x),label='order '+str(order))
plt.legend()

plt.figure()
plt.title("Taylor approximation of order 50 for sine")
x = np.linspace(-20,20,200)
plt.ylim(-1.2,1.2)  
plt.plot(x,taylor_sin(50,x))
plt.plot(x,np.zeros(len(x)),':')   # ':' plots a dotted line
plt.show()

The first positive *x* for which sin(*x*) = 0 is a bit over 3. 
Let's use Newton-Raphson to identify it. The answer is remarkably close to a number we're very familiar with.

In [ ]:
def newton_raphson(x0,poly,tol=1E-14):
    x = x0
    deriv = poly.deriv()
    for n in range(100):
        y = poly(x)
        if abs(y) < tol:
            break
        if deriv(x) == 0:
            x += np.sqrt(tol)  # move x slightly
        x -= y / deriv(x)
    return x

for order in range(10,60,10):
    print('order',order,': zero at',newton_raphson(3,taylor_sin(order)))
print('    Compare to π =',np.pi)

The spacing of zeros gets very regular as the order increases. 
Where the sine approximation is very close to 0, the cosine approximation is very close to 1 or -1.

In [ ]:
outer = 5
z = np.pi * np.linspace(-outer,outer,2*outer+1)
print('\nValues of sine approximation at multiples of π from ',-outer,' to ',outer)
print(taylor_sin(50)(z))
print('\nValues of cosine approximation at multiples of π from ',-outer,' to ',outer)
print(taylor_cos(50)(z))

This code displays resonance for a spring using Taylor series,
as discussed in the math lesson.
However, it is difficult to generate many cycles without very long 
Taylor series, which in turn threaten rounding errors.
Better computations take advantage of cyclical elements in 
sine and cosine, which we will explore in Level 7.

In [ ]:
def resonance(c, b, x0, v0, order, x=None):
    coef = np.zeros(order+1)
    coef[0] = x0
    coef[1] = v0
    cosine_coef = b 
    for k in range(2,order+1,2):
        cosine_coef *= -c / (k * (k-1))
        coef[k] = (-c * coef[k-2]) / (k * (k-1)) + cosine_coef
    for k in range(3,order + 1,2): 
        coef[k] = -c * coef[k-2] / (k*(k-1))
    if x is None:
        return P(coef)
    if isinstance(x,list):
        x = np.array(x)
    return P(coef)(x)

c,b,x0,v0 = 30,1,2,1
time = np.linspace(0,5,500)
x = resonance(c, b, x0, v0, 80)(time)
plt.plot(time,x)
plt.show()